# SC4002 Group Assignment: Joint Intent Classification and Slot Filling (MASSIVE en-US)
## Stage 2 candidate M4: Co-Interactive model (Yuxuan)

**Group:** `<GROUP ID>`

**Stage 1 (Sections 0-13) is copied unchanged from `full_version_resa.ipynb`**, so M4 is trained on the same data pipeline, scored by the same evaluation functions and compared against the same tuned baseline. No Stage 1 code cell was edited. Stage 2 (Sections 14-25) builds and evaluates the M4 Co-Interactive model.

Run it top to bottom (a GPU is recommended). It downloads MASSIVE 1.0 and GloVe 6B (about 820 MB) on first run.

| Section | Content |
|---|---|
| 0-13 | **Stage 1** (copied unchanged): setup, data, preprocessing, shared evaluation, GloVe, baseline BiLSTM, training, checkpoint, test, seed variance, results |
| 14 | Stage 1 diagnosis (validation only) |
| 15 | Why M4: from diagnosis to design |
| 16 | Shared Stage 2 inputs: character ids, the BIO-constrained CRF, GloVe 840B vectors |
| 17 | M4 model definition, parameter count, memorisation check, learning-rate stability check |
| 18 | M4 hyperparameter sweep (validation only) |
| 19 | M4 official training run and checkpoint selection |
| 20 | M4 final evaluation from the saved checkpoint |
| 21 | M4 seed variance (validation only) |
| 22 | Ablations (validation only) |
| 23 | Diagnosis: Stage 1 vs M4 (validation only) |
| 24 | Error cases (validation only) |
| 25 | Final results table |

**Data hygiene.** All design and hyperparameter decisions use the training and validation splits only. The test split is evaluated once per stage, on the frozen checkpoint.

## 0. Setup and configuration

In [ ]:
%pip install -q seqeval==1.2.2

In [ ]:
import json
import platform
import random
import re
import shutil
import tarfile
import time
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seqeval.metrics
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Deterministic cuDNN kernels so that reruns with the same seed give the same numbers.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print(f"python      {platform.python_version()}")
print(f"torch       {torch.__version__}")
print(f"numpy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"device      {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

In [ ]:
# ---- Paths ----
DATA_DIR = Path("data")
CKPT_DIR = Path("checkpoints")
FIG_DIR = Path("figures")
for d in (DATA_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(exist_ok=True)

MASSIVE_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
MASSIVE_TAR = DATA_DIR / "amazon-massive-dataset-1.0.tar.gz"
MASSIVE_JSONL = DATA_DIR / "en-US.jsonl"

# GloVe 6B (Wikipedia + Gigaword, uncased). The second URL is a mirror, tried if the first fails.
GLOVE_URLS = [
    "https://nlp.stanford.edu/data/glove.6B.zip",
    "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
]
GLOVE_ZIP = DATA_DIR / "glove.6B.zip"
GLOVE_TXT = DATA_DIR / "glove.6B.300d.txt"
EMB_DIM = 300

# ---- Assignment constants ----
EXPECTED_SPLITS = {"train": 11_514, "dev": 2_033, "test": 2_974}
PARAM_CAP = 20_000_000

# ---- Stage 1 training recipe ----
STAGE1_DEFAULT = dict(
    hidden=256,          # BiLSTM hidden size per direction
    layers=2,            # BiLSTM layers
    dropout=0.5,         # embedding, between-layer and pre-head dropout
    word_dropout=0.1,    # prob. of replacing a training token with <unk>
    lr=1e-3,
    weight_decay=1e-2,   # AdamW, applied to weight matrices only (not embeddings or biases)
    batch_size=32,
    grad_clip=5.0,
)
MAX_EPOCHS = 50
PATIENCE = 8             # early stopping on the validation selection score
EVAL_BATCH_SIZE = 256

# One setting changed at a time from the default. Selection is on validation only.
STAGE1_SWEEP = [
    ("default", {}),
    ("lr=5e-4", {"lr": 5e-4}),
    ("lr=2e-3", {"lr": 2e-3}),
    ("dropout=0.3", {"dropout": 0.3}),
    ("word_dropout=0", {"word_dropout": 0.0}),
]
SWEEP_SEED = 42
OFFICIAL_SEED = 42       # fixed in advance
EXTRA_SEEDS = [43, 44]   # for mean +/- std on validation

## 1. Load and verify the dataset

We use the original MASSIVE 1.0 release, file `1.0/data/en-US.jsonl`, and split it by the existing `partition` field. No examples are added, removed or moved between splits.

In [ ]:
def download(url, dest):
    """Download url to dest (via a .part file so an interrupted download is never mistaken for a complete one)."""
    dest = Path(dest)
    if dest.exists():
        print(f"found {dest}")
        return dest
    tmp = dest.with_name(dest.name + ".part")
    print(f"downloading {url}")
    t0 = time.time()
    with urllib.request.urlopen(url, timeout=60) as response, open(tmp, "wb") as f:
        shutil.copyfileobj(response, f, length=1 << 20)
    tmp.rename(dest)
    print(f"  saved {dest} ({dest.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.0f}s)")
    return dest


if not MASSIVE_JSONL.exists():
    download(MASSIVE_URL, MASSIVE_TAR)
    with tarfile.open(MASSIVE_TAR, "r:gz") as tar:
        member = next(m for m in tar.getmembers() if m.name.endswith("1.0/data/en-US.jsonl"))
        with tar.extractfile(member) as src, open(MASSIVE_JSONL, "wb") as dst:
            shutil.copyfileobj(src, dst)
    print(f"extracted {member.name} -> {MASSIVE_JSONL}")

with open(MASSIVE_JSONL, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("partitions in file:", dict(Counter(r["partition"] for r in records)))
splits = {name: [r for r in records if r["partition"] == name] for name in EXPECTED_SPLITS}
for name, expected in EXPECTED_SPLITS.items():
    print(f"{name:5s}: {len(splits[name]):,} examples")
    assert len(splits[name]) == expected, f"{name}: expected {expected}, got {len(splits[name])}"
assert sum(len(v) for v in splits.values()) == 16_521
assert all(r["locale"] == "en-US" for r in records)

prefix_mismatch = sum(not r["intent"].startswith(r["scenario"] + "_") for r in splits["train"])
print(f"train intents whose prefix is not their scenario: {prefix_mismatch}")
print("\nexample record:")
print({k: splits["train"][0][k] for k in ("id", "scenario", "intent", "utt", "annot_utt")})

## 2. Text processing: slot parsing, tokenization, BIO labels

**Gold slots come directly from `annot_utt`.** Each `[type : span]` is parsed and aligned to `utt` character by character (ignoring whitespace), giving `(type, span text, char_start, char_end)`. The model only ever sees `utt`.

**Tokens keep their character offsets.** A token is labelled `B-type`/`I-type` if it overlaps a gold slot. When the model predicts a span over tokens `a..b`, its text is read back from `utt[start_a : end_b]`. This makes the slot metric independent of the tokenizer: gold pairs never change, whatever tokenizer a model uses.

**Decoding rule for invalid BIO sequences** (fixed for every experiment): a span starts at any `B-X` or at an `I-X` that does not continue an `X` span, and extends over the following `I-X` tags. This is the CoNLL convention, also used by `seqeval`.

Two tokenizers are defined; Section 5 picks one using the training split only.

In [ ]:
SLOT_RE = re.compile(r"\[(\S+) : ([^\]]+)\]")


def norm_ws(s):
    return " ".join(s.split())


def parse_annot(utt, annot):
    """Gold slots as [(type, span_text, char_start, char_end)], with offsets into utt."""
    slots, i, pos = [], 0, 0

    def skip_space(i):
        while i < len(utt) and utt[i].isspace():
            i += 1
        return i

    def consume(segment, i):
        for ch in segment:
            if ch.isspace():
                continue
            i = skip_space(i)
            if i >= len(utt) or utt[i].lower() != ch.lower():
                raise ValueError(f"annot_utt does not match utt:\n  utt:   {utt!r}\n  annot: {annot!r}")
            i += 1
        return i

    for m in SLOT_RE.finditer(annot):
        i = consume(annot[pos:m.start()], i)
        start = skip_space(i)
        i = consume(m.group(2), start)
        slots.append((m.group(1), norm_ws(utt[start:i]), start, i))
        pos = m.end()
    i = consume(annot[pos:], i)
    if utt[i:].strip():
        raise ValueError(f"utt has extra text after alignment: {utt!r} / {annot!r}")
    return slots


TOKENIZERS = {
    # split on whitespace only
    "whitespace": re.compile(r"\S+"),
    # Penn-Treebank-style: split clitics ('s, n't, 're, ...) and punctuation, like GloVe 6B's own tokenization
    "ptb_split": re.compile(r"\w+(?=n't\b)|n't\b|'\w+|\w+|[^\w\s]"),
}


def tokenize(text, mode):
    """[(token, char_start, char_end)]"""
    return [(m.group(), m.start(), m.end()) for m in TOKENIZERS[mode].finditer(text)]


def bio_tags(tokens, slots):
    tags = ["O"] * len(tokens)
    for stype, _, s, e in slots:
        first = True
        for k, (_, ts, te) in enumerate(tokens):
            if ts < e and te > s and tags[k] == "O":
                tags[k] = ("B-" if first else "I-") + stype
                first = False
    return tags


def tags_to_spans(tags):
    """BIO tags -> [(type, first_token, last_token)] using the decoding rule above."""
    spans, k = [], 0
    while k < len(tags):
        if tags[k] == "O":
            k += 1
            continue
        stype, start = tags[k][2:], k
        k += 1
        while k < len(tags) and tags[k] == "I-" + stype:
            k += 1
        spans.append((stype, start, k - 1))
    return spans


def spans_to_pairs(spans, tokens, utt):
    """Token spans -> Counter of (type, span text), text read from utt via character offsets."""
    return Counter((stype, norm_ws(utt[tokens[a][1]:tokens[b][2]])) for stype, a, b in spans)


# quick illustration
_u = "wake me up at five am this week"
_slots = parse_annot(_u, "wake me up at [time : five am] [date : this week]")
_toks = tokenize(_u, "whitespace")
print(_slots)
print(list(zip([t[0] for t in _toks], bio_tags(_toks, _slots))))

## 3. Shared evaluation functions

Used, unchanged, for every experiment in both stages.

- **Intent accuracy** = 100 x correct intents / number of utterances.
- **Slot micro F1**: for each utterance, gold is the multiset of `(type, span text)` pairs from `annot_utt` and the prediction is the multiset decoded from the predicted tags. A true positive needs both type and span to match exactly. TP, FP and FN are pooled over all utterances, then F1 = 100 x 2TP / (2TP + FP + FN).
- **Exact match** (diagnostic only, not graded): intent correct and the slot multiset identical.
- **Selection score** for checkpoints and hyperparameters, the same in both stages: (intent accuracy + slot micro F1) / 2.

In [ ]:
def intent_accuracy(gold, pred):
    assert len(gold) == len(pred) and len(gold) > 0
    return 100.0 * sum(g == p for g, p in zip(gold, pred)) / len(gold)


def slot_micro_prf(gold_pairs, pred_pairs):
    """gold_pairs, pred_pairs: lists (one per utterance) of Counter[(type, span_text)]."""
    assert len(gold_pairs) == len(pred_pairs)
    tp = fp = fn = 0
    for g, p in zip(gold_pairs, pred_pairs):
        hit = sum((g & p).values())
        tp += hit
        fp += sum(p.values()) - hit
        fn += sum(g.values()) - hit
    denom = 2 * tp + fp + fn
    return dict(
        f1=100.0 * 2 * tp / denom if denom else 100.0,
        precision=100.0 * tp / (tp + fp) if tp + fp else 0.0,
        recall=100.0 * tp / (tp + fn) if tp + fn else 0.0,
        tp=tp, fp=fp, fn=fn,
    )


def exact_match(gold_intents, pred_intents, gold_pairs, pred_pairs):
    hits = sum(gi == pi and gp == pp for gi, pi, gp, pp in zip(gold_intents, pred_intents, gold_pairs, pred_pairs))
    return 100.0 * hits / len(gold_intents)


def selection_score(intent_acc, slot_f1):
    return (intent_acc + slot_f1) / 2


def score_predictions(examples, pred_intents, pred_tags):
    """examples: preprocessed examples (Section 5). pred_tags: list of BIO tag lists aligned to each example's tokens."""
    gold_int = [ex["intent"] for ex in examples]
    gold_pairs = [ex["gold_pairs"] for ex in examples]
    pred_pairs = [spans_to_pairs(tags_to_spans(t), ex["tokens"], ex["utt"]) for ex, t in zip(examples, pred_tags)]
    acc = intent_accuracy(gold_int, pred_intents)
    prf = slot_micro_prf(gold_pairs, pred_pairs)
    return dict(
        intent_acc=acc,
        slot_f1=prf["f1"],
        slot_precision=prf["precision"],
        slot_recall=prf["recall"],
        exact_match=exact_match(gold_int, pred_intents, gold_pairs, pred_pairs),
        score=selection_score(acc, prf["f1"]),
    )

In [ ]:
# ---- Unit tests for the evaluation functions ----
def _pred(utt, tags, mode="whitespace"):
    toks = tokenize(utt, mode)
    assert len(toks) == len(tags), (toks, tags)
    return spans_to_pairs(tags_to_spans(tags), toks, utt)


def _f1(gold, pred):
    return round(slot_micro_prf(gold, pred)["f1"], 4)


U = "wake me up at five am this week"
G = Counter({("time", "five am"): 1, ("date", "this week"): 1})

# 1. perfect prediction
assert _f1([G], [_pred(U, "O O O O B-time I-time B-date I-date".split())]) == 100.0
# 2. boundary error: 'five' instead of 'five am' -> TP=1, FP=1, FN=1
assert _f1([G], [_pred(U, "O O O O B-time O B-date I-date".split())]) == 50.0
# 3. right span, wrong type -> TP=1, FP=1, FN=1
assert _f1([G], [_pred(U, "O O O O B-date I-date B-date I-date".split())]) == 50.0
# 4. missed slot -> TP=1, FP=0, FN=1
assert _f1([G], [_pred(U, "O O O O B-time I-time O O".split())]) == round(100 * 2 / 3, 4)
# 5. spurious slot -> TP=2, FP=1, FN=0
assert _f1([G], [_pred(U, "B-person O O O B-time I-time B-date I-date".split())]) == 80.0
# 6. stray I- tags start a span (CoNLL rule)
assert _f1([G], [_pred(U, "O O O O I-time I-time I-date I-date".split())]) == 100.0
assert tags_to_spans(["B-time", "I-date", "I-date"]) == [("time", 0, 0), ("date", 1, 2)]
# 7. the same slot twice in one utterance is counted twice
U2 = "alarms at five and five"
G2 = Counter({("time", "five"): 2})
assert _f1([G2], [_pred(U2, "O O B-time O B-time".split())]) == 100.0
assert _f1([G2], [_pred(U2, "O O B-time O O".split())]) == round(100 * 2 / 3, 4)
# 8. no slots in gold or prediction
assert _f1([Counter()], [Counter()]) == 100.0
# 9. micro pooling over utterances (not an average of per-utterance F1)
assert _f1([G, Counter()], [_pred(U, "O O O O B-time I-time B-date I-date".split()), Counter({("date", "x"): 1})]) == 80.0
# 10. a slot glued to a clitic: whitespace tokens cannot represent it, ptb_split tokens can
U3 = "what is on today's schedule"
S3 = parse_annot(U3, "what is on [date : today]'s schedule")
assert S3 == [("date", "today", 11, 16)]
for mode, expected in [("whitespace", 0.0), ("ptb_split", 100.0)]:
    toks = tokenize(U3, mode)
    pred = spans_to_pairs(tags_to_spans(bio_tags(toks, S3)), toks, U3)
    assert _f1([Counter({("date", "today"): 1})], [pred]) == expected, mode
# 11. intent accuracy and exact match
assert intent_accuracy(["a", "b", "c", "d"], ["a", "b", "x", "d"]) == 75.0
assert exact_match(["a", "b"], ["a", "b"], [G, Counter()], [G, Counter({("date", "x"): 1})]) == 50.0
# 12. cross-check with seqeval on a case where both definitions apply
_gold_tags = [["O", "O", "O", "O", "B-time", "I-time", "B-date", "I-date"]]
_pred_tags = [["O", "O", "O", "O", "B-time", "O", "B-date", "I-date"]]
assert round(100 * seqeval.metrics.f1_score(_gold_tags, _pred_tags), 4) == _f1([G], [_pred(U, _pred_tags[0])])

print("all evaluation unit tests passed")

## 4. Pretrained word vectors (GloVe 6B, 300d)

Word vectors are the only pretrained component allowed. GloVe 6B is uncased, which matches the lowercased MASSIVE text. Only the rows for our training vocabulary are kept (Section 6), so the embedding table stays small.

In [ ]:
if not GLOVE_TXT.exists():
    for url in GLOVE_URLS:
        try:
            download(url, GLOVE_ZIP)
            break
        except Exception as e:
            print(f"  failed: {e}")
    else:
        raise RuntimeError(f"Could not download GloVe. Put glove.6B.300d.txt at {GLOVE_TXT} manually.")
    with zipfile.ZipFile(GLOVE_ZIP) as z, z.open("glove.6B.300d.txt") as src, open(GLOVE_TXT, "wb") as dst:
        shutil.copyfileobj(src, dst, length=1 << 20)
    print(f"extracted {GLOVE_TXT}")

with open(GLOVE_TXT, encoding="utf-8") as f:
    GLOVE_WORDS = {line.split(" ", 1)[0] for line in f}
print(f"GloVe vocabulary: {len(GLOVE_WORDS):,} words")

## 5. Preprocessing: tokenizer choice, vocabulary and label maps

**Tokenizer choice (training split only).** For each tokenizer we measure:
1. the *slot ceiling*: slot F1 obtained by decoding the gold BIO tags back to `(type, span)` pairs. Below 100 means some gold slot boundary falls inside a token and can never be predicted;
2. GloVe coverage of the training tokens.

We pick the tokenizer with the highest ceiling, breaking ties by coverage.

**Vocabulary** is built from training tokens only (lowercased), plus `<pad>` and `<unk>`. **Label maps** (intents, BIO tags) are built from the training split.

In [ ]:
PAD, UNK, IGNORE = 0, 1, -100


def build_examples(split_records, mode):
    out = []
    for r in split_records:
        slots = parse_annot(r["utt"], r["annot_utt"])
        toks = tokenize(r["utt"], mode)
        assert len(toks) > 0, r
        out.append(dict(
            id=r["id"], utt=r["utt"], intent=r["intent"], scenario=r["scenario"],
            tokens=toks, words=[t[0].lower() for t in toks],
            tags=bio_tags(toks, slots),
            gold_pairs=Counter((s[0], s[1]) for s in slots),
        ))
    return out


def gold_ceiling(examples):
    decoded = [spans_to_pairs(tags_to_spans(ex["tags"]), ex["tokens"], ex["utt"]) for ex in examples]
    return slot_micro_prf([ex["gold_pairs"] for ex in examples], decoded)["f1"]


rows = []
for mode in TOKENIZERS:
    ex_train = build_examples(splits["train"], mode)
    counts = Counter(w for ex in ex_train for w in ex["words"])
    rows.append(dict(
        tokenizer=mode,
        train_slot_ceiling=gold_ceiling(ex_train),
        vocab_types=len(counts),
        glove_type_coverage=100 * sum(w in GLOVE_WORDS for w in counts) / len(counts),
        glove_token_coverage=100 * sum(c for w, c in counts.items() if w in GLOVE_WORDS) / sum(counts.values()),
    ))
tok_df = pd.DataFrame(rows)
display(tok_df.round(2))

best_row = max(rows, key=lambda r: (round(r["train_slot_ceiling"], 6), round(r["glove_token_coverage"], 6)))
TOKENIZER = best_row["tokenizer"]
print(f"chosen tokenizer: {TOKENIZER}")

In [ ]:
EXAMPLES = {name: build_examples(recs, TOKENIZER) for name, recs in splits.items()}

# Pipeline checks (labels are only used to verify preprocessing, not to tune anything).
for name in ("train", "dev"):
    print(f"{name:5s} slot ceiling with {TOKENIZER}: {gold_ceiling(EXAMPLES[name]):.2f}")
missed = [ex for ex in EXAMPLES["train"]
          if spans_to_pairs(tags_to_spans(ex["tags"]), ex["tokens"], ex["utt"]) != ex["gold_pairs"]]
print(f"train utterances where gold tags cannot reproduce gold slots: {len(missed)}")
for ex in missed[:5]:
    print("  ", ex["utt"], "|", dict(ex["gold_pairs"]))

# ---- Vocabulary (training split only) ----
word_counts = Counter(w for ex in EXAMPLES["train"] for w in ex["words"])
ITOS = ["<pad>", "<unk>"] + [w for w, _ in word_counts.most_common()]
STOI = {w: i for i, w in enumerate(ITOS)}
V = len(ITOS)

# ---- Label maps (training split only) ----
ID2INTENT = sorted({ex["intent"] for ex in EXAMPLES["train"]})
INTENT2ID = {x: i for i, x in enumerate(ID2INTENT)}
SLOT_TYPES = sorted({t[2:] for ex in EXAMPLES["train"] for t in ex["tags"] if t != "O"})
ID2TAG = ["O"] + [f"{p}-{s}" for s in SLOT_TYPES for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(ID2TAG)}
N_INTENTS, N_TAGS = len(ID2INTENT), len(ID2TAG)

print(f"\nvocabulary size V = {V:,} (including <pad>, <unk>)")
print(f"intents = {N_INTENTS} (assignment: 60), slot types = {len(SLOT_TYPES)} (assignment: 55), BIO tags = {N_TAGS}")
if (N_INTENTS, len(SLOT_TYPES)) != (60, 55):
    print("note: label counts differ from the assignment description; labels unseen in training cannot be predicted")

lens = np.array([len(ex["tokens"]) for ex in EXAMPLES["train"]])
print(f"train tokens per utterance: mean {lens.mean():.1f}, max {lens.max()}")
dev_tokens = [w for ex in EXAMPLES["dev"] for w in ex["words"]]
print(f"dev tokens not in the training vocabulary: {100 * sum(w not in STOI for w in dev_tokens) / len(dev_tokens):.2f}%")


def encode(ex):
    return dict(
        ids=[STOI.get(w, UNK) for w in ex["words"]],
        tag_ids=[TAG2ID.get(t, IGNORE) for t in ex["tags"]],
        intent_id=INTENT2ID.get(ex["intent"], IGNORE),
    )


DATASETS = {name: [(i, encode(ex)) for i, ex in enumerate(exs)] for name, exs in EXAMPLES.items()}

ex0 = EXAMPLES["train"][0]
print("\nexample:", ex0["utt"], "->", ex0["intent"])
print(list(zip(ex0["words"], ex0["tags"])))

## 6. Embedding matrix

Rows for training words found in GloVe are initialised with their GloVe vectors. `<unk>` and training words missing from GloVe get small random vectors (same scale as GloVe); `<pad>` is all zeros. The matrix is fine-tuned during training, so `<unk>` (trained through word dropout) and the random rows can learn. Words outside the training vocabulary map to `<unk>` at test time; no external lookup is used at inference.

In [ ]:
glove_vecs = {}
with open(GLOVE_TXT, encoding="utf-8") as f:
    for line in f:
        word, rest = line.rstrip("\n").split(" ", 1)
        if word in STOI:
            glove_vecs[word] = np.asarray(rest.split(" "), dtype=np.float32)

found = np.stack(list(glove_vecs.values()))
assert found.shape[1] == EMB_DIM
rng = np.random.default_rng(0)
EMB_MATRIX = rng.normal(0.0, found.std(), size=(V, EMB_DIM)).astype(np.float32)
for w, vec in glove_vecs.items():
    EMB_MATRIX[STOI[w]] = vec
EMB_MATRIX[PAD] = 0.0

n_types = V - 2
n_tok = sum(word_counts.values())
print(f"training word types found in GloVe: {len(glove_vecs):,} / {n_types:,} ({100 * len(glove_vecs) / n_types:.2f}%)")
print(f"training tokens covered by GloVe:  {100 * sum(c for w, c in word_counts.items() if w in glove_vecs) / n_tok:.2f}%")
print(f"embedding matrix: {EMB_MATRIX.shape}")

## 7. Stage 1 model: joint BiLSTM

One shared encoder, two heads, one loss. A single forward pass produces both outputs.

| Part | Details |
|---|---|
| Input | token ids of `utt` (tokenizer from Section 5), lowercased; unknown words map to `<unk>` |
| Embedding | V x 300, initialised from GloVe 6B, fine-tuned; dropout |
| Word dropout | during training only, each real token is replaced by `<unk>` with probability `word_dropout` |
| Encoder | 2-layer bidirectional LSTM, 256 units per direction, dropout between layers, packed sequences so padding is never read |
| Intent head | max-pool over the real (non-padding) time steps -> dropout -> Linear(512, #intents) |
| Slot head | per token: dropout -> Linear(512, #BIO tags), softmax |
| Outputs | one intent label; one BIO tag per token, decoded to `[type : span]` pairs with the rule in Section 2 |
| Loss | CE(intent) + CE(slot tags), padding positions ignored |
| Optimizer | AdamW, lr 1e-3 (tuned in Section 9), weight decay 0.01 on weight matrices, gradient-norm clipping 5, batch size 32 |
| Selection | best validation (intent accuracy + slot micro F1) / 2, early stopping with patience 8, at most 50 epochs |

In [ ]:
class JointBiLSTM(nn.Module):
    """Stage 1: shared BiLSTM encoder, max-pooled intent head, per-token softmax slot head."""

    def __init__(self, emb_matrix, n_intents, n_tags, hidden=256, layers=2, dropout=0.5,
                 word_dropout=0.1, **_unused):
        super().__init__()
        self.word_dropout = word_dropout
        # torch.tensor copies, so training never modifies the shared EMB_MATRIX used by later runs
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=PAD)
        self.emb_dropout = nn.Dropout(dropout)
        self.encoder = nn.LSTM(emb_matrix.shape[1], hidden, num_layers=layers, bidirectional=True,
                               batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head_dropout = nn.Dropout(dropout)
        self.intent_head = nn.Linear(2 * hidden, n_intents)
        self.slot_head = nn.Linear(2 * hidden, n_tags)
        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE)

    def forward(self, batch):
        ids, lengths, mask = batch["ids"], batch["lengths"], batch["mask"]
        if self.training and self.word_dropout > 0:
            drop = (torch.rand(ids.shape, device=ids.device) < self.word_dropout) & mask
            ids = ids.masked_fill(drop, UNK)
        x = self.emb_dropout(self.embedding(ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.encoder(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))
        pooled = h.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(dim=1).values
        return {
            "intent": self.intent_head(self.head_dropout(pooled)),
            "slot": self.slot_head(self.head_dropout(h)),
        }

    def loss(self, out, batch):
        intent_loss = self.ce(out["intent"], batch["intents"])
        slot_loss = self.ce(out["slot"].reshape(-1, out["slot"].size(-1)), batch["tags"].reshape(-1))
        return {"total": intent_loss + slot_loss, "intent": intent_loss, "slot": slot_loss}

    def decode(self, out, batch):
        intents = out["intent"].argmax(-1).tolist()
        tags = out["slot"].argmax(-1).tolist()
        return intents, [t[:n] for t, n in zip(tags, batch["lengths"].tolist())]


def build_stage1(cfg):
    return JointBiLSTM(EMB_MATRIX, N_INTENTS, N_TAGS, **cfg)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def param_breakdown(model):
    rows = [(name, sum(p.numel() for p in m.parameters())) for name, m in model.named_children()]
    rows = [(n, c) for n, c in rows if c > 0]
    total, trainable = count_params(model)
    df = pd.DataFrame(rows + [("TOTAL", total), ("trainable", trainable)], columns=["component", "parameters"])
    df["parameters"] = df["parameters"].map("{:,}".format)
    return df, total, trainable


_m = build_stage1(STAGE1_DEFAULT)
_df, _total, _trainable = param_breakdown(_m)
display(_df)
print(f"total parameters in the Stage 1 pipeline: {_total:,} (cap {PARAM_CAP:,})")
assert _total <= PARAM_CAP
del _m

## 8. Training and evaluation utilities

Shared by both stages. A model only has to implement `forward(batch)`, `loss(out, batch)` and `decode(out, batch)`, so Stage 2 models (for example with a CRF decoder) plug into the same loop and the same evaluation.

In [ ]:
def collate(batch):
    idx = [b[0] for b in batch]
    items = [b[1] for b in batch]
    lengths = torch.tensor([len(x["ids"]) for x in items])
    T = int(lengths.max())
    ids = torch.full((len(items), T), PAD, dtype=torch.long)
    tags = torch.full((len(items), T), IGNORE, dtype=torch.long)
    for i, x in enumerate(items):
        n = len(x["ids"])
        ids[i, :n] = torch.tensor(x["ids"])
        tags[i, :n] = torch.tensor(x["tag_ids"])
    return dict(
        idx=torch.tensor(idx), ids=ids, lengths=lengths,
        mask=torch.arange(T)[None, :] < lengths[:, None],
        tags=tags, intents=torch.tensor([x["intent_id"] for x in items]),
    )


def make_loader(split, batch_size, shuffle, seed=None):
    gen = None
    if shuffle:
        gen = torch.Generator()
        gen.manual_seed(seed)
    return DataLoader(DATASETS[split], batch_size=batch_size, shuffle=shuffle, collate_fn=collate, generator=gen)


def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


@torch.no_grad()
def evaluate(model, split):
    """Returns (metrics dict incl. mean loss, predicted intents, predicted tag lists) in example order."""
    model.eval()
    n = len(EXAMPLES[split])
    pred_int, pred_tags = [None] * n, [None] * n
    loss_sum, count = 0.0, 0
    for batch in make_loader(split, EVAL_BATCH_SIZE, shuffle=False):
        b = to_device(batch)
        out = model(b)
        loss_sum += model.loss(out, b)["total"].item() * len(batch["idx"])
        count += len(batch["idx"])
        ints, tag_seqs = model.decode(out, b)
        for i, it, ts in zip(batch["idx"].tolist(), ints, tag_seqs):
            pred_int[i] = ID2INTENT[it]
            pred_tags[i] = [ID2TAG[t] for t in ts]
    metrics = score_predictions(EXAMPLES[split], pred_int, pred_tags)
    metrics["loss"] = loss_sum / count
    return metrics, pred_int, pred_tags


def make_optimizer(model, cfg):
    """AdamW. Weight decay only on weight matrices: never on embedding tables, biases or CRF scores."""
    emb_params = {id(p) for m in model.modules() if isinstance(m, nn.Embedding) for p in m.parameters()}
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 and id(p) not in emb_params and not name.startswith("crf") else no_decay).append(p)
    return torch.optim.AdamW([
        {"params": decay, "weight_decay": cfg["weight_decay"]},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=cfg["lr"])


def train_one_epoch(model, loader, opt, grad_clip):
    model.train()
    sums, n = Counter(), 0
    for batch in loader:
        b = to_device(batch)
        losses = model.loss(model(b), b)
        opt.zero_grad(set_to_none=True)
        losses["total"].backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()
        bs = len(batch["idx"])
        for k, v in losses.items():
            sums[k] += v.item() * bs
        n += bs
    return {k: v / n for k, v in sums.items()}


def save_checkpoint(path, model, cfg, seed, row, stage):
    torch.save(dict(
        stage=stage, model_class=type(model).__name__, model_state=model.state_dict(),
        cfg=cfg, seed=seed, epoch=row["epoch"], dev_metrics=dict(row),
        vocab=ITOS, intents=ID2INTENT, tags=ID2TAG, tokenizer=TOKENIZER,
    ), path)


def load_checkpoint(path, build_fn):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    assert ck["vocab"] == ITOS and ck["intents"] == ID2INTENT and ck["tags"] == ID2TAG and ck["tokenizer"] == TOKENIZER
    model = build_fn(ck["cfg"]).to(DEVICE)
    model.load_state_dict(ck["model_state"])
    model.eval()
    return model, ck


def run_training(build_fn, cfg, seed, ckpt_path=None, verbose=True, stage="stage1"):
    """Train on the training split; select the epoch with the best validation selection score."""
    set_seed(seed)
    model = build_fn(cfg).to(DEVICE)
    opt = make_optimizer(model, cfg)
    # Optional (Stage 2): halve the learning rate after 3 epochs without a better validation score.
    sched = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
             if cfg.get("lr_schedule", "none") == "plateau" else None)
    loader = make_loader("train", cfg["batch_size"], shuffle=True, seed=seed)
    history, best, bad, t_start = [], None, 0, time.time()
    if verbose:
        print(f"{'epoch':>5} | {'train loss':>10} | {'dev loss':>8} | {'dev intent':>10} | {'dev slot F1':>11} | {'dev score':>9} | {'time':>6}")
    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        tr = train_one_epoch(model, loader, opt, cfg["grad_clip"])
        dev, _, _ = evaluate(model, "dev")
        row = dict(
            epoch=epoch,
            train_loss=tr["total"], train_intent_loss=tr["intent"], train_slot_loss=tr["slot"],
            dev_loss=dev["loss"], dev_intent_acc=dev["intent_acc"], dev_slot_f1=dev["slot_f1"],
            dev_exact_match=dev["exact_match"], dev_score=dev["score"],
            epoch_seconds=time.time() - t0, elapsed_seconds=time.time() - t_start,
            lr=opt.param_groups[0]["lr"],
        )
        history.append(row)
        if sched is not None:
            sched.step(row["dev_score"])
        improved = best is None or row["dev_score"] > best["dev_score"]
        if improved:
            best, bad = dict(row), 0
            if ckpt_path is not None:
                save_checkpoint(ckpt_path, model, cfg, seed, row, stage)
        else:
            bad += 1
        if verbose:
            print(f"{epoch:>5} | {row['train_loss']:>10.4f} | {row['dev_loss']:>8.4f} | {row['dev_intent_acc']:>10.2f} | "
                  f"{row['dev_slot_f1']:>11.2f} | {row['dev_score']:>9.2f} | {row['epoch_seconds']:>5.1f}s {'*' if improved else ''}")
        if bad >= PATIENCE:
            if verbose:
                print(f"early stopping: no improvement for {PATIENCE} epochs")
            break
    return model, history, best


def plot_curves(history, best_epoch, title, path):
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["epoch"], h["train_loss"], label="train loss")
    axes[0].plot(h["epoch"], h["dev_loss"], label="validation loss")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (intent CE + slot CE)"); axes[0].legend()
    axes[1].plot(h["epoch"], h["dev_intent_acc"], label="validation intent accuracy")
    axes[1].plot(h["epoch"], h["dev_slot_f1"], label="validation slot micro F1")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("%"); axes[1].legend()
    for ax in axes:
        ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=1)
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    fig.suptitle(f"{title} (dashed line: selected epoch {best_epoch})")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

## 9. Stage 1 hyperparameter sweep (validation only)

A small sweep so that the baseline is properly tuned: one setting is changed at a time from the default, one seed each. The configuration with the best validation selection score is used for the official run. No test data is touched here.

In [ ]:
sweep_rows = []
for name, override in STAGE1_SWEEP:
    cfg = {**STAGE1_DEFAULT, **override}
    t0 = time.time()
    m, hist, best = run_training(build_stage1, cfg, seed=SWEEP_SEED, verbose=False)
    del m
    sweep_rows.append(dict(
        run=name, best_epoch=best["epoch"], epochs_run=len(hist),
        dev_intent_acc=best["dev_intent_acc"], dev_slot_f1=best["dev_slot_f1"], dev_score=best["dev_score"],
        minutes=(time.time() - t0) / 60,
    ))
    print(f"{name:16s} best epoch {best['epoch']:>2}/{len(hist):>2} | intent {best['dev_intent_acc']:.2f} | "
          f"slot F1 {best['dev_slot_f1']:.2f} | score {best['dev_score']:.2f} | {(time.time() - t0) / 60:.1f} min")

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.round(2))
sweep_df.to_csv("stage1_sweep.csv", index=False)

best_run = sweep_df.loc[sweep_df["dev_score"].idxmax(), "run"]
STAGE1_CFG = {**STAGE1_DEFAULT, **dict(STAGE1_SWEEP)[best_run]}
print(f"selected configuration: {best_run}")
print(STAGE1_CFG)

## 10. Stage 1 official training run and checkpoint selection

Selected configuration, seed fixed in advance. Every epoch logs training and validation losses, validation metrics and elapsed time. The checkpoint with the highest validation selection score is saved (`*` marks an improvement).

In [ ]:
STAGE1_CKPT = CKPT_DIR / "stage1_best.pt"
stage1_train_model, stage1_history, stage1_best = run_training(
    build_stage1, STAGE1_CFG, seed=OFFICIAL_SEED, ckpt_path=STAGE1_CKPT, verbose=True, stage="stage1")
del stage1_train_model
pd.DataFrame(stage1_history).to_csv("stage1_history.csv", index=False)

print(f"\nselected checkpoint: epoch {stage1_best['epoch']} (of {len(stage1_history)} run)")
print(f"validation selection score: {stage1_best['dev_score']:.2f} "
      f"(intent accuracy {stage1_best['dev_intent_acc']:.2f}, slot micro F1 {stage1_best['dev_slot_f1']:.2f})")
print(f"checkpoint file: {STAGE1_CKPT}")
print(f"total training time: {stage1_history[-1]['elapsed_seconds'] / 60:.1f} min")

In [ ]:
plot_curves(stage1_history, stage1_best["epoch"], "Stage 1 learning curves", FIG_DIR / "stage1_curves.png")

## 11. Stage 1 final evaluation from the saved checkpoint

A fresh model is built and the saved checkpoint is loaded from disk. Without further training, it is evaluated on the validation and test splits. The validation score must equal the best score logged during training.

In [ ]:
stage1_model, stage1_ck = load_checkpoint(STAGE1_CKPT, build_stage1)
stage1_dev, stage1_dev_int, stage1_dev_tags = evaluate(stage1_model, "dev")
stage1_test, _, _ = evaluate(stage1_model, "test")

print(f"checkpoint: {STAGE1_CKPT} (epoch {stage1_ck['epoch']})")
print(f"validation score logged during training: {stage1_ck['dev_metrics']['dev_score']:.6f}")
print(f"validation score after reloading:       {stage1_dev['score']:.6f}")
assert abs(stage1_dev["score"] - stage1_ck["dev_metrics"]["dev_score"]) < 1e-6

# Cross-check of our slot F1 against seqeval on token-level BIO tags (validation).
_seqeval_f1 = 100 * seqeval.metrics.f1_score([ex["tags"] for ex in EXAMPLES["dev"]], stage1_dev_tags)
print(f"validation slot F1: ours {stage1_dev['slot_f1']:.2f}, seqeval {_seqeval_f1:.2f}")

stage1_eval_df = pd.DataFrame([
    {"split": "validation", **{k: stage1_dev[k] for k in ("intent_acc", "slot_f1", "slot_precision", "slot_recall", "exact_match")}},
    {"split": "test", **{k: stage1_test[k] for k in ("intent_acc", "slot_f1", "slot_precision", "slot_recall", "exact_match")}},
])
display(stage1_eval_df.round(2))
print(f"\nStage 1 TEST: intent accuracy {stage1_test['intent_acc']:.2f}%, slot micro F1 {stage1_test['slot_f1']:.2f}%")

## 12. Stage 1 seed variance (validation only)

The same configuration trained with two more seeds, to show how much of a validation difference is just noise. Only validation scores are reported; these runs are not used for the official checkpoint.

In [ ]:
seed_rows = [dict(seed=OFFICIAL_SEED, best_epoch=stage1_best["epoch"], dev_intent_acc=stage1_best["dev_intent_acc"],
                  dev_slot_f1=stage1_best["dev_slot_f1"], dev_score=stage1_best["dev_score"])]
for s in EXTRA_SEEDS:
    m, hist, best = run_training(build_stage1, STAGE1_CFG, seed=s, verbose=False)
    del m
    seed_rows.append(dict(seed=s, best_epoch=best["epoch"], dev_intent_acc=best["dev_intent_acc"],
                          dev_slot_f1=best["dev_slot_f1"], dev_score=best["dev_score"]))
stage1_seeds_df = pd.DataFrame(seed_rows)
display(stage1_seeds_df.round(2))
for col in ("dev_intent_acc", "dev_slot_f1", "dev_score"):
    print(f"{col:15s} mean {stage1_seeds_df[col].mean():.2f} +/- {stage1_seeds_df[col].std(ddof=1):.2f} (std over {len(stage1_seeds_df)} seeds)")

## 13. Results table

Generated from the variables above (no numbers are typed by hand). "Number of epochs" is the epoch of the selected checkpoint; the number of epochs actually run before early stopping is printed below the table. Stage 2 rows will be appended in the Stage 2 section.

In [ ]:
def results_rows(name, model, dev, test, best_epoch):
    total, trainable = count_params(model)
    return [
        {"Model version": name, "Split": "Validation", "Total # parameters": f"{total:,}",
         "Total # trainable parameters": f"{trainable:,}", "Intent accuracy (%)": f"{dev['intent_acc']:.2f}",
         "Slot micro-F1 (%)": f"{dev['slot_f1']:.2f}", "Number of epochs": str(best_epoch)},
        {"Model version": name, "Split": "Test", "Total # parameters": "Same as above",
         "Total # trainable parameters": "Same as above", "Intent accuracy (%)": f"{test['intent_acc']:.2f}",
         "Slot micro-F1 (%)": f"{test['slot_f1']:.2f}", "Number of epochs": "Same as above"},
    ]


RESULTS = results_rows("Stage-1 system", stage1_model, stage1_dev, stage1_test, stage1_ck["epoch"])
display(pd.DataFrame(RESULTS))
print(f"Stage 1: selected epoch {stage1_ck['epoch']}, {len(stage1_history)} epochs run (early stopping, patience {PATIENCE})")

with open("results_stage1.json", "w") as f:
    json.dump(dict(rows=RESULTS, dev=stage1_dev, test=stage1_test, cfg=STAGE1_CFG,
                   epochs_run=len(stage1_history), selected_epoch=stage1_ck["epoch"]), f, indent=2)

# Stage 2: M4 Co-Interactive model

## 14. Stage 1 diagnosis (validation split only)

Before changing the model, we measure where Stage 1 fails, using the **validation** predictions of the selected Stage 1 checkpoint (Section 11). The diagnosis functions are the same as in `full_version_resa.ipynb`, so every Stage 2 candidate is diagnosed the same way. They are run again on M4 in Section 23.

In [ ]:
from collections import defaultdict

pd.set_option("display.max_colwidth", None)  # show full measure names in the diagnosis tables
ERROR_KINDS =["wrong type", "boundary", "boundary + type", "missed", "spurious"]

# Training-split statistics used by the analyses (training labels only).
TRAIN_INTENT_SLOTS = defaultdict(set)
TRAIN_SLOT_FREQ = Counter()
for _ex in EXAMPLES["train"]:
    for _stype, _, _ in tags_to_spans(_ex["tags"]):
        TRAIN_INTENT_SLOTS[_ex["intent"]].add(_stype)
        TRAIN_SLOT_FREQ[_stype] += 1


def _overlap(a, b):
    return a[1] <= b[2] and b[1] <= a[2]


def _pct(a, b):
    return 100.0 * a / b if b else float("nan")


def slot_error_breakdown(examples, pred_tags):
    """Each gold span is correct, wrong type (same words), boundary (same type, overlapping words),
    boundary + type, or missed (no overlapping prediction). Each predicted span overlapping no gold span is spurious."""
    c = Counter()
    for ex, pt in zip(examples, pred_tags):
        gold, pred = tags_to_spans(ex["tags"]), tags_to_spans(pt)
        pred_set, gold_set = set(pred), set(gold)
        for g in gold:
            if g in pred_set:
                c["correct"] += 1
                continue
            over = [p for p in pred if _overlap(p, g)]
            if any(p[1:] == g[1:] for p in over):
                c["wrong type"] += 1
            elif any(p[0] == g[0] for p in over):
                c["boundary"] += 1
            elif over:
                c["boundary + type"] += 1
            else:
                c["missed"] += 1
            if len(over) >= 2:
                c["split into pieces"] += 1
        for p in pred:
            if p not in gold_set and not any(_overlap(p, g) for g in gold):
                c["spurious"] += 1
    return c


def invalid_bio(tag_lists):
    """Number of I-X tags that do not continue an X span, and number of utterances containing one."""
    bad_tags = bad_utts = 0
    for tags in tag_lists:
        prev, bad = "O", 0
        for t in tags:
            if t.startswith("I-") and prev[2:] != t[2:]:
                bad += 1
            prev = t
        bad_tags += bad
        bad_utts += bad > 0
    return bad_tags, bad_utts


def subset_scores(examples, pred_int, pred_tags, mask):
    idx = [i for i, keep in enumerate(mask) if keep]
    if not idx:
        return dict(n=0, intent_acc=float("nan"), slot_f1=float("nan"))
    m = score_predictions([examples[i] for i in idx], [pred_int[i] for i in idx], [pred_tags[i] for i in idx])
    return dict(n=len(idx), intent_acc=m["intent_acc"], slot_f1=m["slot_f1"])


def span_recall_by_oov(examples, pred_tags):
    """Share of gold spans predicted exactly, for spans with / without a word outside the training vocabulary."""
    hit, tot = Counter(), Counter()
    for ex, pt in zip(examples, pred_tags):
        pred = set(tags_to_spans(pt))
        for g in tags_to_spans(ex["tags"]):
            key = "oov" if any(ex["words"][k] not in STOI for k in range(g[1], g[2] + 1)) else "in_vocab"
            tot[key] += 1
            hit[key] += g in pred
    return {k: dict(n=tot[k], recall=_pct(hit[k], tot[k])) for k in ("oov", "in_vocab")}


def slot_intent_consistency(examples, pred_int, pred_tags):
    """Predicted spans whose type never occurs with the predicted intent in training, and how many of them are wrong."""
    n = bad = bad_wrong = 0
    for ex, pi, pt in zip(examples, pred_int, pred_tags):
        gold = set(tags_to_spans(ex["tags"]))
        for p in tags_to_spans(pt):
            n += 1
            if p[0] not in TRAIN_INTENT_SLOTS[pi]:
                bad += 1
                bad_wrong += p not in gold
    return n, bad, bad_wrong


def per_slot_type_f1(examples, pred_tags):
    tp, fp, fn = Counter(), Counter(), Counter()
    for ex, pt in zip(examples, pred_tags):
        g = ex["gold_pairs"]
        p = spans_to_pairs(tags_to_spans(pt), ex["tokens"], ex["utt"])
        for (t, _), k in (g & p).items():
            tp[t] += k
        for (t, _), k in (p - g).items():
            fp[t] += k
        for (t, _), k in (g - p).items():
            fn[t] += k
    rows = []
    for t in sorted(set(tp) | set(fp) | set(fn)):
        d = 2 * tp[t] + fp[t] + fn[t]
        rows.append(dict(slot_type=t, train_count=TRAIN_SLOT_FREQ[t], dev_gold=tp[t] + fn[t],
                         tp=tp[t], fp=fp[t], fn=fn[t], f1=100.0 * 2 * tp[t] / d if d else 100.0))
    return pd.DataFrame(rows)


def diagnose(name, pred_int, pred_tags, split="dev", show=True):
    exs = EXAMPLES[split]
    assert split != "test", "diagnosis is for the validation split only"
    m = score_predictions(exs, pred_int, pred_tags)
    br = slot_error_breakdown(exs, pred_tags)
    n_gold = sum(len(tags_to_spans(ex["tags"])) for ex in exs)
    bad_tags, bad_utts = invalid_bio(pred_tags)
    has_oov = [any(w not in STOI for w in ex["words"]) for ex in exs]
    oov_yes = subset_scores(exs, pred_int, pred_tags, has_oov)
    oov_no = subset_scores(exs, pred_int, pred_tags, [not x for x in has_oov])
    span_oov = span_recall_by_oov(exs, pred_tags)
    n_pred, incons, incons_wrong = slot_intent_consistency(exs, pred_int, pred_tags)
    intent_errs = [(ex["intent"], p) for ex, p in zip(exs, pred_int) if ex["intent"] != p]
    same_scen = sum(g.split("_")[0] == p.split("_")[0] for g, p in intent_errs)
    int_ok = [ex["intent"] == p for ex, p in zip(exs, pred_int)]
    s_ok = subset_scores(exs, pred_int, pred_tags, int_ok)
    s_bad = subset_scores(exs, pred_int, pred_tags, [not x for x in int_ok])
    n_err = sum(br[k] for k in ERROR_KINDS)

    summary = [
        ("intent accuracy / slot F1 / exact match", f"{m['intent_acc']:.2f} / {m['slot_f1']:.2f} / {m['exact_match']:.2f}"),
        ("slot precision / recall", f"{m['slot_precision']:.2f} / {m['slot_recall']:.2f}"),
        ("predicted slot spans / gold slot spans", f"{n_pred:,} / {n_gold:,}"),
        ("slot errors (all kinds)", f"{n_err:,}"),
        ("invalid I- tags (utterances affected)", f"{bad_tags:,} ({bad_utts:,})"),
        ("gold spans split into several predicted pieces", f"{br['split into pieces']:,}"),
        ("utterances with an OOV word: n, intent acc, slot F1",
         f"{oov_yes['n']:,}, {oov_yes['intent_acc']:.2f}, {oov_yes['slot_f1']:.2f}"),
        ("utterances without OOV words: n, intent acc, slot F1",
         f"{oov_no['n']:,}, {oov_no['intent_acc']:.2f}, {oov_no['slot_f1']:.2f}"),
        ("gold spans with / without an OOV word: exact recall",
         f"{span_oov['oov']['recall']:.2f} (n={span_oov['oov']['n']:,}) / {span_oov['in_vocab']['recall']:.2f} (n={span_oov['in_vocab']['n']:,})"),
        ("predicted spans with a type never seen with the predicted intent (share wrong)",
         f"{incons:,} of {n_pred:,} ({_pct(incons_wrong, incons):.1f}% wrong)"),
        ("intent errors: total, within the same scenario", f"{len(intent_errs):,}, {_pct(same_scen, len(intent_errs)):.1f}%"),
        ("slot F1 when intent correct / wrong", f"{s_ok['slot_f1']:.2f} (n={s_ok['n']:,}) / {s_bad['slot_f1']:.2f} (n={s_bad['n']:,})"),
    ]
    out = dict(name=name, metrics=m, breakdown=br, n_gold=n_gold, n_pred=n_pred, invalid_tags=bad_tags,
               incons=incons, intent_errors=intent_errs, per_type=per_slot_type_f1(exs, pred_tags), summary=summary)
    if show:
        print(f"=== {name}: diagnosis on {split} ({len(exs):,} utterances) ===")
        display(pd.DataFrame(summary, columns=["measure", "value"]))
        bd = pd.DataFrame([dict(kind=k, count=br[k], share_of_errors=_pct(br[k], n_err)) for k in ERROR_KINDS])
        display(bd.round(1))
        print("most confused intent pairs (gold -> predicted):")
        display(pd.DataFrame(Counter(intent_errs).most_common(10), columns=["gold -> predicted", "count"]))
        print("weakest slot types (at least 10 gold spans in validation):")
        pt = out["per_type"]
        display(pt[pt["dev_gold"] >= 10].sort_values("f1").head(12).round(2))
    return out


def show_errors(pred_int, pred_tags, split="dev", n=25, seed=0):
    exs = EXAMPLES[split]
    assert split != "test"

    def fmt(c):
        return " ".join(f"[{t} : {s}]" for (t, s), k in sorted(c.items()) for _ in range(k)) or "(none)"

    wrong = [i for i, ex in enumerate(exs)
             if ex["intent"] != pred_int[i]
             or spans_to_pairs(tags_to_spans(pred_tags[i]), ex["tokens"], ex["utt"]) != ex["gold_pairs"]]
    print(f"{len(wrong):,} of {len(exs):,} validation utterances have at least one error; a fixed random sample of {min(n, len(wrong))}:\n")
    for i in random.Random(seed).sample(wrong, min(n, len(wrong))):
        ex = exs[i]
        pred = spans_to_pairs(tags_to_spans(pred_tags[i]), ex["tokens"], ex["utt"])
        flag = "" if ex["intent"] == pred_int[i] else "   <- intent wrong"
        print(f"- {ex['utt']}")
        print(f"    intent  gold {ex['intent']:24s} pred {pred_int[i]}{flag}")
        print(f"    slots   gold {fmt(ex['gold_pairs'])}")
        print(f"            pred {fmt(pred)}")


def plot_error_breakdown(diags, path):
    x = np.arange(len(ERROR_KINDS))
    w = 0.8 / len(diags)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for k, d in enumerate(diags):
        bars = ax.bar(x + (k - (len(diags) - 1) / 2) * w, [d["breakdown"][e] for e in ERROR_KINDS], w, label=d["name"])
        ax.bar_label(bars, fontsize=8)
    ax.set_xticks(x, ERROR_KINDS)
    ax.set_ylabel("validation slot errors (count)")
    ax.set_title("Slot errors by kind on the validation split")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

In [ ]:
stage1_diag = diagnose("Stage 1", stage1_dev_int, stage1_dev_tags)
plot_error_breakdown([stage1_diag], FIG_DIR / "stage1_errors.png")

## 15. Why M4: from diagnosis to design

In Stage 1 the two tasks share only the BiLSTM states: the intent head never sees which slots were found, and the slot head never sees which intent was chosen. Three measures in Section 14 show how much the tasks depend on each other:

- **slot F1 when the intent is correct vs wrong**: slot quality collapses on utterances whose intent is wrong;
- **predicted slot types never seen with the predicted intent**: slots that contradict the intent are mostly wrong;
- **intent errors within the same scenario**: many intent errors confuse related intents, where the slot words (a time, a song name, a place) are the deciding evidence.

A one-way design (intent → slots, like the M1 gate) can use the intent to fix slots, but cannot use slot evidence to fix the intent. **M4 lets information flow both ways**: slot evidence such as "five am is a time" strengthens the intent `alarm_set`, and the intent sharpens the slot decisions.

| Part | M4 | Weakness it targets |
|---|---|---|
| Word input | GloVe vectors for the training vocabulary (6B as in Stage 1, or 840B, chosen on validation; fine-tuned) + **character CNN** (50-d chars, 100 filters of width 3) | words outside the training vocabulary |
| Encoder | BiLSTM, Stage 1 recipe | - |
| **Label attention** | every word is compared with learned embeddings of the intents and of the BIO tags; the attention-weighted label vectors are added to the word state. This gives two streams: an intent stream H_I and a slot stream H_S | gives each task its own explicit representation |
| **Co-interactive layers** (x L) | cross-attention: the intent stream queries the slot stream and the slot stream queries the intent stream, then one feed-forward block reads both streams; residual connections and LayerNorm | information flows slots → intent and intent → slots |
| Intent decoder | max-pool H_I over the real tokens → linear | |
| Slot decoder | linear → **CRF** with BIO constraints, decoded with Viterbi | boundary errors, invalid BIO sequences |
| Loss | CE(intent) + lambda x CRF negative log-likelihood per token, lambda = 1 | |
| Optimizer | Stage 1 recipe (AdamW, weight decay on weight matrices, clipping, batch size, early stopping) except the learning rate: 5e-4, because 1e-3 is too high for the attention layers (Section 17); halved after 3 epochs without a better validation score | |

**Reference.** L. Qin, T. Liu, W. Che, B. Kang, S. Zhao, T. Liu. *A Co-Interactive Transformer for Joint Slot Filling and Intent Detection.* ICASSP 2021 ([arXiv:2010.03880](https://arxiv.org/abs/2010.03880)). We follow its structure (shared BiLSTM, label attention, co-interactive attention plus feed-forward layers, max-pooled intent, CRF slots); sizes and training settings are ours, tuned on validation.

**Protocol (validation only until Section 20).** A memorisation check, then a small sweep that changes one setting at a time with one seed (as in Stage 1), then the official run with the fixed seed, the reload-and-test step, two extra seeds, and ablations with 3 seeds each that remove one part of M4 at a time.

## 16. Shared Stage 2 inputs: character ids, the BIO-constrained CRF and GloVe 840B

The character vocabulary, the collate function with character ids and the CRF are the same as in `full_version_resa.ipynb`. The CRF is checked against brute-force enumeration of every tag sequence, and its BIO constraints make an invalid sequence (an `I-X` that does not continue an `X` span) impossible to predict.

In [ ]:
import itertools
import math
import torch.nn.functional as F

# ---- Character vocabulary (training words only) ----
MAX_WORD_LEN = 20
CHAR_ITOS = ["<pad>", "<unk>"] + sorted({ch for ex in EXAMPLES["train"] for w in ex["words"] for ch in w})
CHAR2ID = {c: i for i, c in enumerate(CHAR_ITOS)}
N_CHARS = len(CHAR_ITOS)
for name, data in DATASETS.items():
    for i, enc in data:
        enc["char_ids"] = [[CHAR2ID.get(ch, 1) for ch in w[:MAX_WORD_LEN]] for w in EXAMPLES[name][i]["words"]]
print(f"character vocabulary: {N_CHARS} symbols (including <pad>, <unk>)")


def collate_with_chars(batch):
    """Same as the Stage 1 collate, plus a (batch, tokens, characters) tensor of character ids."""
    out = collate_stage1(batch)
    items = [b[1] for b in batch]
    L = max(len(c) for x in items for c in x["char_ids"])
    chars = torch.zeros((len(items), out["ids"].size(1), L), dtype=torch.long)
    for i, x in enumerate(items):
        for t, c in enumerate(x["char_ids"]):
            chars[i, t, :len(c)] = torch.tensor(c)
    out["chars"] = chars
    return out


if "collate_stage1" not in globals():   # guard so that re-running this cell is safe
    collate_stage1 = collate
collate = collate_with_chars   # used by make_loader from here on; Stage 1 models simply ignore "chars"

In [ ]:
class CRF(nn.Module):
    """Linear-chain conditional random field over tag sequences.
    allowed[i, j] = 1 if tag j may follow tag i; start_allowed[j] = 1 if tag j may start a sequence.
    Forbidden moves get a -1e4 score in both training and decoding, so they are never predicted."""

    def __init__(self, n_tags, allowed=None, start_allowed=None):
        super().__init__()
        self.n_tags = n_tags
        self.transitions = nn.Parameter(torch.empty(n_tags, n_tags).uniform_(-0.1, 0.1))
        self.start = nn.Parameter(torch.empty(n_tags).uniform_(-0.1, 0.1))
        self.end = nn.Parameter(torch.empty(n_tags).uniform_(-0.1, 0.1))
        allowed = torch.ones(n_tags, n_tags) if allowed is None else allowed.float()
        start_allowed = torch.ones(n_tags) if start_allowed is None else start_allowed.float()
        self.register_buffer("trans_penalty", (1 - allowed) * -1e4)
        self.register_buffer("start_penalty", (1 - start_allowed) * -1e4)

    def nll(self, emissions, tags, mask):
        """Negative log-likelihood of the gold tags, averaged over real tokens (same scale as token cross-entropy)."""
        B, T, K = emissions.shape
        tags = torch.where(mask & (tags >= 0), tags, torch.zeros_like(tags))
        oh = F.one_hot(tags, K).to(emissions.dtype)          # one-hot instead of gather: deterministic on GPU
        maskf = mask.to(emissions.dtype)
        trans, start = self.transitions + self.trans_penalty, self.start + self.start_penalty
        gold = ((emissions * oh).sum(-1) * maskf).sum(1) + (oh[:, 0] * start).sum(-1)
        gold = gold + (torch.einsum("btk,kl,btl->bt", oh[:, :-1], trans, oh[:, 1:]) * maskf[:, 1:]).sum(1)
        last = mask.sum(1) - 1
        gold = gold + (oh[torch.arange(B, device=tags.device), last] * self.end).sum(-1)
        alpha = start + emissions[:, 0]
        for t in range(1, T):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0), dim=1) + emissions[:, t]
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        log_z = torch.logsumexp(alpha + self.end, dim=1)
        return (log_z - gold).sum() / maskf.sum()

    @torch.no_grad()
    def viterbi(self, emissions, mask):
        """Highest-scoring tag sequence for each sentence, shape (batch, tokens)."""
        B, T, K = emissions.shape
        trans = self.transitions + self.trans_penalty
        score = self.start + self.start_penalty + emissions[:, 0]
        same = torch.arange(K, device=emissions.device).expand(B, K)
        backpointers = []
        for t in range(1, T):
            best, idx = (score.unsqueeze(2) + trans.unsqueeze(0)).max(dim=1)
            m = mask[:, t].unsqueeze(1)
            score = torch.where(m, best + emissions[:, t], score)
            backpointers.append(torch.where(m, idx, same))
        last = (score + self.end).argmax(1)
        path = [last]
        for idx in reversed(backpointers):
            last = idx.gather(1, last.unsqueeze(1)).squeeze(1)
            path.append(last)
        return torch.stack(path[::-1], dim=1)


def bio_constraints(tag_names):
    """I-X may only follow B-X or I-X, and may not start a sentence."""
    K = len(tag_names)
    allowed, start = torch.zeros(K, K), torch.zeros(K)
    for j, tj in enumerate(tag_names):
        start[j] = not tj.startswith("I-")
        for i, ti in enumerate(tag_names):
            allowed[i, j] = (not tj.startswith("I-")) or (ti != "O" and ti[2:] == tj[2:])
    return allowed, start


# ---- Unit tests: compare with brute-force enumeration of every tag sequence ----
import itertools

def _brute(crf, em, length, gold=None):
    trans = crf.transitions + crf.trans_penalty
    start = crf.start + crf.start_penalty
    scores = {}
    for path in itertools.product(range(crf.n_tags), repeat=length):
        s = start[path[0]] + em[0, path[0]] + crf.end[path[-1]]
        for t in range(1, length):
            s = s + trans[path[t - 1], path[t]] + em[t, path[t]]
        scores[path] = s
    log_z = torch.logsumexp(torch.stack(list(scores.values())), 0)
    best = max(scores, key=lambda p: scores[p].item())
    nll = (log_z - scores[tuple(gold)]).item() if gold is not None else None
    return best, nll

torch.manual_seed(0)
_names = ["O", "B-a", "I-a", "B-b", "I-b"]
for constrained in (False, True):
    _crf = CRF(5, *bio_constraints(_names)) if constrained else CRF(5)
    _em = torch.randn(2, 4, 5)
    _mask = torch.tensor([[1, 1, 1, 1], [1, 1, 1, 0]], dtype=torch.bool)
    _gold = torch.tensor([[1, 2, 0, 3], [3, 4, 0, -100]])
    _vit = _crf.viterbi(_em, _mask)
    total = 0.0
    for b, n in enumerate([4, 3]):
        best, nll = _brute(_crf, _em[b], n, _gold[b, :n].tolist())
        assert tuple(_vit[b, :n].tolist()) == best, (constrained, b)
        total += nll
    assert abs(_crf.nll(_em, _gold, _mask).item() - total / 7) < 1e-4
    if constrained:
        _many = _crf.viterbi(torch.randn(64, 6, 5) * 3, torch.ones(64, 6, dtype=torch.bool))
        assert invalid_bio([[_names[k] for k in row] for row in _many.tolist()]) == (0, 0)
print("CRF unit tests passed (forward algorithm and Viterbi match brute force; constrained decoding never emits invalid BIO)")

### A second word-vector source: GloVe 840B

Stage 1 uses GloVe 6B (6 billion tokens of Wikipedia and news, 400k uncased words). GloVe 840B was trained on 840 billion tokens of web text (Common Crawl, 2.2M cased words). We only copy rows for the **same training vocabulary** as Stage 1, so the embedding table keeps the same size and parameter count; only its starting values change. Words outside the training vocabulary still map to `<unk>`, as in Stage 1.

GloVe 840B is cased. For each (lowercased) training word we take its lowercase entry if it exists, otherwise its most frequent cased spelling (GloVe lists words from most to least frequent). Words missing from 840B get small random vectors, exactly as in Section 6. The trimmed table is cached in `data/` so the 2 GB archive is read only once. Which source M4 uses is decided on validation (Section 18).

In [ ]:
import hashlib
import io

GLOVE840_URL = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip"
GLOVE840_ZIP = DATA_DIR / "glove.840B.300d.zip"
VOCAB_HASH = hashlib.md5("\n".join(ITOS).encode()).hexdigest()[:10]
GLOVE840_CACHE = DATA_DIR / f"glove840b_rows_{VOCAB_HASH}.npz"


def read_glove840(words):
    """Stream the zipped text file once and keep one vector per word in `words` (lowercase)."""
    need, exact, cased = set(words), {}, {}
    with zipfile.ZipFile(GLOVE840_ZIP) as z, z.open("glove.840B.300d.txt") as fb:
        for line in io.TextIOWrapper(fb, encoding="utf-8"):
            head = line[:line.find(" ")]
            low = head.lower()
            if low not in need or low in exact or (head != low and low in cased):
                continue
            parts = line.rstrip().rsplit(" ", EMB_DIM)
            if len(parts) != EMB_DIM + 1 or parts[0] != head:   # a few lines hold words that contain spaces
                continue
            (exact if head == low else cased)[low] = np.asarray(parts[1:], dtype=np.float32)
    return {**cased, **exact}


if GLOVE840_CACHE.exists():
    _z = np.load(GLOVE840_CACHE)
    EMB_MATRIX_840B, HAS_840B = _z["matrix"], _z["has_vec"]
    print(f"loaded cached GloVe 840B rows: {GLOVE840_CACHE}")
else:
    download(GLOVE840_URL, GLOVE840_ZIP)
    _t0 = time.time()
    _vecs = read_glove840(ITOS[2:])
    _found = np.stack(list(_vecs.values()))
    _rng = np.random.default_rng(0)
    EMB_MATRIX_840B = _rng.normal(0.0, _found.std(), size=(V, EMB_DIM)).astype(np.float32)
    for w, vec in _vecs.items():
        EMB_MATRIX_840B[STOI[w]] = vec
    EMB_MATRIX_840B[PAD] = 0.0
    HAS_840B = np.array([w in _vecs for w in ITOS])
    np.savez(GLOVE840_CACHE, matrix=EMB_MATRIX_840B, has_vec=HAS_840B)
    print(f"read GloVe 840B in {(time.time() - _t0) / 60:.1f} min; cached the {V:,} rows in {GLOVE840_CACHE}")

EMB_SOURCES = {"glove6b": EMB_MATRIX, "glove840b": EMB_MATRIX_840B}
_has6 = np.array([w in glove_vecs for w in ITOS])
_tok = np.array([word_counts.get(w, 0) for w in ITOS])
_rows = []
for name, has in [("GloVe 6B (Stage 1)", _has6), ("GloVe 840B", HAS_840B)]:
    _rows.append(dict(source=name, training_word_types_found=f"{has[2:].sum():,} / {V - 2:,} ({100 * has[2:].mean():.2f}%)",
                      training_tokens_covered=f"{100 * _tok[has].sum() / _tok.sum():.2f}%",
                      table_parameters=f"{EMB_SOURCES['glove6b' if '6B' in name else 'glove840b'].size:,}"))
display(pd.DataFrame(_rows))
print(f"training words found in 840B but not in 6B: {(HAS_840B & ~_has6)[2:].sum():,}; in 6B but not in 840B: {(_has6 & ~HAS_840B)[2:].sum():,}")

## 17. M4 model definition

`CoInteractiveJoint` implements `forward`, `loss` and `decode`, so it plugs into the Stage 1 training loop and evaluation unchanged. Switches turn parts off for the ablations in Section 22: with `label_attention=False` and `co_layers=0` it is the character CNN + BiLSTM + CRF backbone.

Shapes, for a batch of B utterances with T tokens and d = 2 x hidden:

1. word vectors (B, T, 300) ⊕ character features (B, T, 100) → BiLSTM → H (B, T, d)
2. label attention: A_I = softmax(H E_I^T) over the intents, H_I = H + A_I E_I; the same with the BIO-tag embeddings E_S gives H_S
3. each co-interactive layer: H_I ← LN(H_I + MultiHeadAttention(query H_I, key/value H_S)), H_S ← LN(H_S + MultiHeadAttention(query H_S, key/value H_I)); then F = FFN([H_I ; H_S]), H_I ← LN(H_I + F), H_S ← LN(H_S + F). Padding positions are never attended to.
4. intent logits from max-pooled H_I; slot emission scores from H_S, decoded by the CRF

For the one-way ablation (`slot_to_intent=False`), the intent stream never reads the slot stream: its cross-attention is removed and the slot half of its feed-forward input is replaced by zeros.

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, dim=50, filters=100, kernel=3, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.conv = nn.Conv1d(dim, filters, kernel, padding=kernel // 2)
        self.out_dim = filters

    def forward(self, chars):                       # (B, T, L) character ids
        B, T, L = chars.shape
        flat = chars.reshape(B * T, L)
        x = self.conv(self.dropout(self.emb(flat)).transpose(1, 2))
        x = x.masked_fill((flat == 0).unsqueeze(1), float("-inf"))
        return torch.relu(x.max(dim=2).values).reshape(B, T, -1)   # padding words give all -inf -> 0


class LabelAttention(nn.Module):
    """Compares every word state with a learned embedding of each label and adds the attention-weighted label vectors."""

    def __init__(self, n_labels, d):
        super().__init__()
        self.labels = nn.Parameter(torch.empty(n_labels, d))
        nn.init.xavier_uniform_(self.labels)

    def forward(self, h):                           # (B, T, d)
        a = torch.softmax(h @ self.labels.t(), dim=-1)    # (B, T, n_labels): how much each word looks like each label
        return h + a @ self.labels, a


class CoInteractiveLayer(nn.Module):
    """Cross-attention between the intent and slot streams, then one feed-forward block that reads both streams.
    pre_norm=False: LN(x + f(x)) as in the paper; pre_norm=True: x + f(LN(x)), which trains stably without warm-up."""

    def __init__(self, d, heads, ffn_dim, dropout, slot_to_intent=True, intent_to_slot=True, pre_norm=False):
        super().__init__()
        self.slot_to_intent, self.intent_to_slot, self.pre_norm = slot_to_intent, intent_to_slot, pre_norm
        # intent queries attend to the slot stream / slot queries attend to the intent stream
        self.att_intent = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True) if slot_to_intent else None
        self.att_slot = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True) if intent_to_slot else None
        self.norm_i1, self.norm_s1 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(2 * d, ffn_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d))
        self.norm_i2, self.norm_s2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, hi, hs, pad):                 # hi, hs: (B, T, d); pad: (B, T), True at padding
        # 1. cross-attention (each stream reads the other)
        qi, qs = (self.norm_i1(hi), self.norm_s1(hs)) if self.pre_norm else (hi, hs)
        if self.att_intent is not None:
            hi = hi + self.drop(self.att_intent(qi, qs, qs, key_padding_mask=pad, need_weights=False)[0])
        if self.att_slot is not None:
            hs = hs + self.drop(self.att_slot(qs, qi, qi, key_padding_mask=pad, need_weights=False)[0])
        if not self.pre_norm:
            hi, hs = self.norm_i1(hi), self.norm_s1(hs)
        # 2. one feed-forward block over both streams
        fi, fs = (self.norm_i2(hi), self.norm_s2(hs)) if self.pre_norm else (hi, hs)
        both = self.ffn(torch.cat([fi, fs], dim=-1))
        f_i = both if self.slot_to_intent else self.ffn(torch.cat([fi, torch.zeros_like(fs)], dim=-1))
        f_s = both if self.intent_to_slot else self.ffn(torch.cat([torch.zeros_like(fi), fs], dim=-1))
        hi, hs = hi + self.drop(f_i), hs + self.drop(f_s)
        if not self.pre_norm:
            hi, hs = self.norm_i2(hi), self.norm_s2(hs)
        return hi, hs


class CoInteractiveJoint(nn.Module):
    """M4: GloVe + character CNN -> BiLSTM -> label attention -> co-interactive layers -> max-pooled intent, CRF slots."""

    def __init__(self, n_chars, n_intents, n_tags, hidden=256, layers=2, dropout=0.5, word_dropout=0.1,
                 char_cnn=True, crf=True, label_attention=True, co_layers=2, heads=4, ffn_dim=None, co_dropout=0.2,
                 slot_to_intent=True, intent_to_slot=True, pre_norm=False, slot_loss_weight=1.0, emb="glove6b", **_unused):
        super().__init__()
        self.word_dropout, self.slot_loss_weight = word_dropout, slot_loss_weight
        matrix = EMB_SOURCES[emb]   # same training vocabulary and size for every source; only the starting values differ
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(matrix), freeze=False, padding_idx=PAD)
        self.char_cnn = CharCNN(n_chars, dropout=dropout) if char_cnn else None
        in_dim = matrix.shape[1] + (self.char_cnn.out_dim if char_cnn else 0)
        self.emb_dropout = nn.Dropout(dropout)
        self.encoder = nn.LSTM(in_dim, hidden, num_layers=layers, bidirectional=True, batch_first=True,
                               dropout=dropout if layers > 1 else 0.0)
        d = 2 * hidden
        self.intent_label_att = LabelAttention(n_intents, d) if label_attention else None
        self.slot_label_att = LabelAttention(n_tags, d) if label_attention else None
        self.co_layers = nn.ModuleList([CoInteractiveLayer(d, heads, ffn_dim or d, co_dropout, slot_to_intent, intent_to_slot,
                                                           pre_norm) for _ in range(co_layers)])
        # pre-norm layers leave the streams un-normalised, so they get one final LayerNorm each
        self.final_norm = nn.ModuleList([nn.LayerNorm(d), nn.LayerNorm(d)]) if pre_norm and co_layers > 0 else None
        self.head_dropout = nn.Dropout(dropout)
        self.intent_head = nn.Linear(d, n_intents)
        self.slot_head = nn.Linear(d, n_tags)
        self.crf = CRF(n_tags, *bio_constraints(ID2TAG)) if crf else None
        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE)

    def forward(self, batch):
        ids, lengths, mask = batch["ids"], batch["lengths"], batch["mask"]
        if self.training and self.word_dropout > 0:
            drop = (torch.rand(ids.shape, device=ids.device) < self.word_dropout) & mask
            ids = ids.masked_fill(drop, UNK)
        x = self.embedding(ids)
        if self.char_cnn is not None:
            x = torch.cat([x, self.char_cnn(batch["chars"])], dim=-1)
        x = self.emb_dropout(x)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.encoder(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))
        if self.intent_label_att is not None:
            hi, _ = self.intent_label_att(h)
            hs, _ = self.slot_label_att(h)
        else:
            hi, hs = h, h
        pad = ~mask
        for layer in self.co_layers:
            hi, hs = layer(hi, hs, pad)
        if self.final_norm is not None:
            hi, hs = self.final_norm[0](hi), self.final_norm[1](hs)
        sent = hi.masked_fill(pad.unsqueeze(-1), float("-inf")).max(dim=1).values
        return {"intent": self.intent_head(self.head_dropout(sent)), "slot": self.slot_head(self.head_dropout(hs))}

    def loss(self, out, batch):
        intent_loss = self.ce(out["intent"], batch["intents"])
        if self.crf is not None:
            slot_loss = self.crf.nll(out["slot"], batch["tags"], batch["mask"])
        else:
            slot_loss = self.ce(out["slot"].reshape(-1, out["slot"].size(-1)), batch["tags"].reshape(-1))
        return {"total": intent_loss + self.slot_loss_weight * slot_loss, "intent": intent_loss, "slot": slot_loss}

    def decode(self, out, batch):
        intents = out["intent"].argmax(-1).tolist()
        tags = (self.crf.viterbi(out["slot"], batch["mask"]) if self.crf is not None else out["slot"].argmax(-1)).tolist()
        return intents, [t[:n] for t, n in zip(tags, batch["lengths"].tolist())]


def build_m4(cfg):
    return CoInteractiveJoint(N_CHARS, N_INTENTS, N_TAGS, **cfg)


# ---- M4 default configuration: the Stage 1 recipe (Section 9) plus the M4 parts ----
# The learning rate is 5e-4: the Stage 1 value (1e-3) is too high for the attention layers (stability check below).
M4_BASE = {**STAGE1_CFG, "lr": 5e-4,
           "char_cnn": True, "crf": True, "lr_schedule": "plateau", "slot_loss_weight": 1.0,
           "label_attention": True, "co_layers": 2, "heads": 4, "ffn_dim": None, "co_dropout": 0.2,
           "slot_to_intent": True, "intent_to_slot": True, "pre_norm": False, "emb": "glove6b"}
M4_SEEDS = [OFFICIAL_SEED] + EXTRA_SEEDS

_df, _total, _ = param_breakdown(build_m4(M4_BASE))
display(_df)
print(f"M4 default configuration: {_total:,} parameters (cap {PARAM_CAP:,})")
assert _total <= PARAM_CAP   # the default is the largest configuration swept in Section 18

### Memorisation check

Before any long run: with dropout off, M4 should fit 64 training utterances perfectly. A model that cannot do this has a bug (for example a masking error), and no amount of tuning would fix it.

In [ ]:
def memorisation_check(cfg, n=64, steps=300):
    set_seed(0)
    model = build_m4({**cfg, "dropout": 0.0, "word_dropout": 0.0, "co_dropout": 0.0}).to(DEVICE)
    batch = to_device(collate(DATASETS["train"][:n]))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(steps):
        loss = model.loss(model(batch), batch)["total"]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        ints, tags = model.decode(model(batch), batch)
    m = score_predictions(EXAMPLES["train"][:n], [ID2INTENT[i] for i in ints], [[ID2TAG[t] for t in ts] for ts in tags])
    return loss.item(), m


_t0 = time.time()
_loss, _m = memorisation_check(M4_BASE)
print(f"after 300 full-batch steps on 64 training utterances: loss {_loss:.4f}, intent accuracy {_m['intent_acc']:.2f}, "
      f"slot F1 {_m['slot_f1']:.2f} ({time.time() - _t0:.0f}s)")
assert _m["intent_acc"] == 100.0 and _m["slot_f1"] == 100.0, "M4 cannot memorise 64 utterances: check the model"

### Training-stability check: learning rate for the attention layers

A short run (6 epochs, seed 42) of the default M4 with the Stage 1 learning rate (1e-3) and with 5e-4. Post-norm attention layers trained without warm-up are known to be sensitive to the learning rate; if 1e-3 fails to learn the slots in the first epochs, M4 uses 5e-4. This check uses the validation split only.

In [ ]:
_stab_rows = []
_saved_max_epochs = MAX_EPOCHS
MAX_EPOCHS = 6   # run_training reads this global; restored below
try:
    for _lr in (1e-3, 5e-4):
        _, _hist, _ = run_training(build_m4, {**M4_BASE, "lr": _lr}, seed=OFFICIAL_SEED, verbose=False, stage="m4-check")
        for _h in _hist:
            _stab_rows.append(dict(lr=_lr, epoch=_h["epoch"], train_loss=_h["train_loss"], dev_intent_acc=_h["dev_intent_acc"],
                                   dev_slot_f1=_h["dev_slot_f1"]))
finally:
    MAX_EPOCHS = _saved_max_epochs
_stab = pd.DataFrame(_stab_rows)
display(_stab.pivot(index="epoch", columns="lr", values=["dev_intent_acc", "dev_slot_f1"]).round(2))

## 18. M4 hyperparameter sweep (validation only)

Same protocol as the Stage 1 sweep (Section 9): one setting changed at a time from the default, one seed each, selection on the validation selection score. No test data is touched.

| Change | Question |
|---|---|
| `co_layers=1` | Is one round of interaction enough (fewer parameters, less overfitting)? |
| `pre-norm` | LayerNorm before each block instead of after (the paper's post-norm). Pre-norm is usually more stable for deeper stacks |
| `lr x 0.5` | Is 5e-4 still too high? |
| `GloVe 840B` | Do word vectors trained on 140 times more text help (same vocabulary and size)? |
| `BiLSTM layers=1` | With the co-interactive layers on top, is a second BiLSTM layer still needed? (The paper uses one.) |

In [ ]:
M4_SWEEP = [
    ("default", {}),
    ("co_layers=1", {"co_layers": 1}),
    ("pre-norm", {"pre_norm": True}),
    ("lr x 0.5", {"lr": M4_BASE["lr"] / 2}),
    ("GloVe 840B", {"emb": "glove840b"}),
    ("BiLSTM layers=1", {"layers": 1}),
]

m4_sweep_rows = []
for name, override in M4_SWEEP:
    cfg = {**M4_BASE, **override}
    t0 = time.time()
    m, hist, best = run_training(build_m4, cfg, seed=SWEEP_SEED, verbose=False, stage="m4-dev")
    n_params = count_params(m)[0]
    del m
    m4_sweep_rows.append(dict(
        run=name, params=n_params, best_epoch=best["epoch"], epochs_run=len(hist),
        dev_intent_acc=best["dev_intent_acc"], dev_slot_f1=best["dev_slot_f1"], dev_score=best["dev_score"],
        minutes=(time.time() - t0) / 60,
    ))
    print(f"{name:16s} best epoch {best['epoch']:>2}/{len(hist):>2} | intent {best['dev_intent_acc']:.2f} | "
          f"slot F1 {best['dev_slot_f1']:.2f} | score {best['dev_score']:.2f} | {(time.time() - t0) / 60:.1f} min")

m4_sweep_df = pd.DataFrame(m4_sweep_rows)
display(m4_sweep_df.round(2))
m4_sweep_df.to_csv("m4_sweep.csv", index=False)

m4_best_run = m4_sweep_df.loc[m4_sweep_df["dev_score"].idxmax(), "run"]
M4_CFG = {**M4_BASE, **dict(M4_SWEEP)[m4_best_run]}
print(f"selected configuration: {m4_best_run}")
print({k: v for k, v in M4_CFG.items() if k not in STAGE1_CFG or STAGE1_CFG[k] != v})

## 19. M4 official training run and checkpoint selection

The selected configuration, trained with the same fixed seed as Stage 1. Every epoch logs training and validation losses, validation metrics and elapsed time. The checkpoint with the highest validation selection score is saved (`*` marks an improvement).

In [ ]:
_df, _total, _trainable = param_breakdown(build_m4(M4_CFG))
display(_df)
print(f"total parameters in the M4 pipeline: {_total:,} (cap {PARAM_CAP:,})")
assert _total <= PARAM_CAP

M4_CKPT = CKPT_DIR / "m4_best.pt"
m4_train_model, m4_history, m4_best = run_training(
    build_m4, M4_CFG, seed=OFFICIAL_SEED, ckpt_path=M4_CKPT, verbose=True, stage="stage2-m4")
del m4_train_model
pd.DataFrame(m4_history).to_csv("m4_history.csv", index=False)

print(f"\nselected checkpoint: epoch {m4_best['epoch']} (of {len(m4_history)} run)")
print(f"validation selection score: {m4_best['dev_score']:.2f} "
      f"(intent accuracy {m4_best['dev_intent_acc']:.2f}, slot micro F1 {m4_best['dev_slot_f1']:.2f})")
print(f"checkpoint file: {M4_CKPT}")
print(f"total training time: {m4_history[-1]['elapsed_seconds'] / 60:.1f} min")

In [ ]:
plot_curves(m4_history, m4_best["epoch"], "M4 learning curves", FIG_DIR / "m4_curves.png")

## 20. M4 final evaluation from the saved checkpoint

A fresh model is built and the saved checkpoint is loaded from disk. Without further training, it is evaluated on the validation and test splits. The validation score must equal the best score logged during training; then the test split is evaluated once.

In [ ]:
m4_model, m4_ck = load_checkpoint(M4_CKPT, build_m4)
m4_dev, m4_dev_int, m4_dev_tags = evaluate(m4_model, "dev")
m4_test, _, m4_test_tags = evaluate(m4_model, "test")

print(f"checkpoint: {M4_CKPT} (epoch {m4_ck['epoch']})")
print(f"validation score logged during training: {m4_ck['dev_metrics']['dev_score']:.6f}")
print(f"validation score after reloading:       {m4_dev['score']:.6f}")
assert abs(m4_dev["score"] - m4_ck["dev_metrics"]["dev_score"]) < 1e-6

_seqeval_f1 = 100 * seqeval.metrics.f1_score([ex["tags"] for ex in EXAMPLES["dev"]], m4_dev_tags)
print(f"validation slot F1: ours {m4_dev['slot_f1']:.2f}, seqeval {_seqeval_f1:.2f}")
if M4_CFG["crf"]:
    assert invalid_bio(m4_dev_tags) == (0, 0) and invalid_bio(m4_test_tags) == (0, 0)
    print("constrained CRF: no invalid BIO sequences on validation or test")

_cols = ("intent_acc", "slot_f1", "slot_precision", "slot_recall", "exact_match")
m4_eval_df = pd.DataFrame([
    {"split": "validation", **{k: m4_dev[k] for k in _cols}},
    {"split": "test", **{k: m4_test[k] for k in _cols}},
])
display(m4_eval_df.round(2))
print(f"\nM4 TEST: intent accuracy {m4_test['intent_acc']:.2f}%, slot micro F1 {m4_test['slot_f1']:.2f}%")

## 21. M4 seed variance (validation only)

The selected configuration trained with the two extra Stage 1 seeds. Only validation scores are reported; these runs are not used for the official checkpoint. They also form the "full M4" row of the ablation table.

In [ ]:
M4_RUNS = {}   # (configuration, seed) -> best validation row, filled only by runs in this notebook


def cfg_key(cfg, seed):
    return (tuple(sorted(cfg.items())), seed)


M4_RUNS[cfg_key(M4_CFG, OFFICIAL_SEED)] = dict(m4_best, epochs_run=len(m4_history),
                                               minutes=m4_history[-1]["elapsed_seconds"] / 60)


def run_seeds(name, cfg, seeds=M4_SEEDS):
    rows = []
    for s in seeds:
        key = cfg_key(cfg, s)
        if key not in M4_RUNS:
            t0 = time.time()
            m, hist, best = run_training(build_m4, cfg, seed=s, verbose=False, stage="m4-dev")
            del m
            M4_RUNS[key] = dict(best, epochs_run=len(hist), minutes=(time.time() - t0) / 60)
        rows.append(dict(M4_RUNS[key], seed=s))
    d = pd.DataFrame(rows)
    res = dict(experiment=name, params=count_params(build_m4(cfg))[0],
               intent_mean=d["dev_intent_acc"].mean(), intent_std=d["dev_intent_acc"].std(ddof=1),
               slot_f1_mean=d["dev_slot_f1"].mean(), slot_f1_std=d["dev_slot_f1"].std(ddof=1),
               score_mean=d["dev_score"].mean(), score_std=d["dev_score"].std(ddof=1),
               best_epochs=" / ".join(str(int(e)) for e in d["epoch"]), minutes=d["minutes"].sum())
    print(f"{name:58s} intent {res['intent_mean']:.2f} +/- {res['intent_std']:.2f} | slot F1 {res['slot_f1_mean']:.2f} +/- "
          f"{res['slot_f1_std']:.2f} | score {res['score_mean']:.2f} +/- {res['score_std']:.2f}")
    return res, d


m4_row, m4_seeds_df = run_seeds("M4 (selected configuration)", M4_CFG)
display(m4_seeds_df[["seed", "epoch", "dev_intent_acc", "dev_slot_f1", "dev_score"]].round(2))
for col in ("dev_intent_acc", "dev_slot_f1", "dev_score"):
    print(f"{col:15s} M4 mean {m4_seeds_df[col].mean():.2f} +/- {m4_seeds_df[col].std(ddof=1):.2f} | "
          f"Stage 1 mean {stage1_seeds_df[col].mean():.2f} +/- {stage1_seeds_df[col].std(ddof=1):.2f} (3 seeds each)")

## 22. Ablations (validation only)

Each row removes one part of M4 from the selected configuration and trains the same 3 seeds. The **backbone** row (no label attention, no co-interactive layers) is the character CNN + BiLSTM + CRF model with learning-rate decay: the same architecture as the Stage 2 model selected in `full_version_resa.ipynb`, so the gap between it and M4 is what the co-interactive part adds. The **one-way** row keeps everything but stops slot information from reaching the intent, which tests M4's main idea directly. The last row swaps the word-vector source (GloVe 6B and 840B), with everything else fixed.

As a noise reference, a difference between two 3-seed means is treated as real only if it exceeds 2 x sqrt(2/3) x (Stage 1 seed standard deviation), twice the standard error of the difference.

In [ ]:
STAGE1_SCORE_STD = stage1_seeds_df["dev_score"].std(ddof=1)
NOISE_MARGIN = 2 * math.sqrt(2 / len(M4_SEEDS)) * STAGE1_SCORE_STD
print(f"Stage 1 seed std of the selection score: {STAGE1_SCORE_STD:.3f} -> noise margin {NOISE_MARGIN:.3f}\n")

ABLATIONS = [
    ("backbone only (no label attention, no co-interactive)", {"label_attention": False, "co_layers": 0}),
    ("label attention only (no co-interactive layers)", {"co_layers": 0}),
    ("no label attention", {"label_attention": False}),
    ("one-way: intent -> slot only (no slot -> intent)", {"slot_to_intent": False}),
]
_other_emb = "glove6b" if M4_CFG["emb"] == "glove840b" else "glove840b"
ABLATIONS.append((f"word vectors: {_other_emb} instead of {M4_CFG['emb']}", {"emb": _other_emb}))
abl_rows = [dict(m4_row, change=float("nan"), beyond_noise="")]
print(f"{'M4 (selected configuration)':58s} score {m4_row['score_mean']:.2f} +/- {m4_row['score_std']:.2f}")
for name, change in ABLATIONS:
    r, _ = run_seeds(name, {**M4_CFG, **change})
    r["change"] = r["score_mean"] - m4_row["score_mean"]
    r["beyond_noise"] = "yes" if abs(r["change"]) > NOISE_MARGIN else "no"
    abl_rows.append(r)

m4_ablation_df = pd.DataFrame(abl_rows)
display(m4_ablation_df.round(2))
m4_ablation_df.to_csv("m4_ablations.csv", index=False)

## 23. Diagnosis: Stage 1 vs M4 (validation only)

The Section 14 analyses again, on the M4 validation predictions, with both systems side by side.

In [ ]:
m4_diag = diagnose("M4", m4_dev_int, m4_dev_tags)
plot_error_breakdown([stage1_diag, m4_diag], FIG_DIR / "errors_stage1_vs_m4.png")
compare = pd.DataFrame({
    "measure": [m for m, _ in stage1_diag["summary"]],
    "Stage 1": [v for _, v in stage1_diag["summary"]],
    "M4": [v for _, v in m4_diag["summary"]],
})
display(compare)

## 24. Error cases (validation only)

Utterances are grouped by whether each system gets them fully right (correct intent and identical slot set): fixed by M4, broken by M4, and wrong in both. A fixed random sample of each group is printed with the gold labels and both predictions, as material for the error analysis in the report.

In [ ]:
def fmt_slots(c):
    return " ".join(f"[{t} : {s}]" for (t, s), k in sorted(c.items()) for _ in range(k)) or "(none)"


def pred_pairs(ex, tags):
    return spans_to_pairs(tags_to_spans(tags), ex["tokens"], ex["utt"])


_dev = EXAMPLES["dev"]
_ok1 = [ex["intent"] == pi and pred_pairs(ex, pt) == ex["gold_pairs"]
        for ex, pi, pt in zip(_dev, stage1_dev_int, stage1_dev_tags)]
_ok4 = [ex["intent"] == pi and pred_pairs(ex, pt) == ex["gold_pairs"]
        for ex, pi, pt in zip(_dev, m4_dev_int, m4_dev_tags)]
ERROR_GROUPS = {
    "fixed by M4 (Stage 1 wrong, M4 right)": [i for i in range(len(_dev)) if not _ok1[i] and _ok4[i]],
    "broken by M4 (Stage 1 right, M4 wrong)": [i for i in range(len(_dev)) if _ok1[i] and not _ok4[i]],
    "wrong in both": [i for i in range(len(_dev)) if not _ok1[i] and not _ok4[i]],
}
display(pd.DataFrame([dict(group=g, utterances=len(ix)) for g, ix in ERROR_GROUPS.items()]))

for g, ix in ERROR_GROUPS.items():
    print(f"\n=== {g}: {len(ix)} utterances, fixed random sample of {min(6, len(ix))} ===")
    for i in random.Random(0).sample(ix, min(6, len(ix))):
        ex = _dev[i]
        print(f"- {ex['utt']}")
        print(f"    gold     {ex['intent']:24s} {fmt_slots(ex['gold_pairs'])}")
        print(f"    Stage 1  {stage1_dev_int[i]:24s} {fmt_slots(pred_pairs(ex, stage1_dev_tags[i]))}")
        print(f"    M4       {m4_dev_int[i]:24s} {fmt_slots(pred_pairs(ex, m4_dev_tags[i]))}")

## 25. Final results table

Generated from the variables above (no numbers are typed by hand), in the format required by the brief. "Number of epochs" is the epoch of each selected checkpoint.

In [ ]:
FINAL_RESULTS = results_rows("Stage-1 system", stage1_model, stage1_dev, stage1_test, stage1_ck["epoch"]) + \
                results_rows("Stage-2 system (M4)", m4_model, m4_dev, m4_test, m4_ck["epoch"])
display(pd.DataFrame(FINAL_RESULTS))
print(f"Stage 1: selected epoch {stage1_ck['epoch']}, {len(stage1_history)} epochs run; "
      f"M4: selected epoch {m4_ck['epoch']}, {len(m4_history)} epochs run (early stopping, patience {PATIENCE})")

with open("results_m4.json", "w") as f:
    json.dump(dict(rows=FINAL_RESULTS, stage1=dict(dev=stage1_dev, test=stage1_test, cfg=STAGE1_CFG),
                   m4=dict(dev=m4_dev, test=m4_test, cfg=M4_CFG, sweep_choice=m4_best_run)), f, indent=2, default=str)